# Notebook 1 — Zero-shot Llama 3 8B (base and Instruct) on Mohler  ·  GPU: T4

Scores all 2,273 Mohler answers with the two untuned models, using the same prompt that fine-tuning will use (R4's "untuned Llama 3 8B under the same prompt" baseline). Also measures inference latency and GPU memory (R1-9).

Scoring reads the model's next-token probabilities over "0"–"10" in one forward pass: predicted score = probability-weighted mean. No sampling and no text parsing, so results are exactly reproducible.

**Before running:** upload `llm_utils.py` to `MyDrive/asag_revision/`. Runtime → Change runtime type → T4 GPU. Resumable: if the session ends, re-run all cells. Expected time: ~30–45 min.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
ROOT = '/content/drive/MyDrive/asag_revision'
sys.path.insert(0, ROOT)
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
!pip -q install -U bitsandbytes accelerate peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 52.2 MB/s eta 0:00:00


In [3]:
import json, time, gc, platform
import numpy as np, pandas as pd, torch, transformers, bitsandbytes
from tqdm.auto import tqdm
from sklearn.linear_model import LinearRegression
from asag_utils import *
from llm_utils import *

versions = {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__,
            'bitsandbytes': bitsandbytes.__version__, 'gpu': torch.cuda.get_device_name(0)}
mo = pd.read_csv(f'{ROOT}/data/mohler_prepared.csv')
prompts = make_prompts(mo, 'mohler')
print(len(mo), 'answers'); print(prompts[0])

2273 answers
You are grading a student's short answer. Compare the student answer with the reference answer and assign a score from 0 (completely incorrect) to 10 (completely correct). Judge meaning, not wording: paraphrases and synonyms are acceptable, but an answer that contradicts the reference answer (for example through negation) is incorrect.

### Question:
What is the role of a prototype program in problem solving?

### Reference Answer:
To simulate the behaviour of portions of the desired software product.

### Student Answer:
High risk problems are address in the prototype program to make sure that the program is feasible.  A prototype may also be used to show a company that the software can be possibly programmed.

### Score (0-10):



## 1. Score every answer with each model (resumable, saves every 256 answers)

In [4]:
MODELS = {'llama3_base_zs': (BASE_MODEL, False), 'llama3_instruct_zs': (INSTRUCT_MODEL, True)}
CHUNK = 256
latency = []

for tag, (name, chat) in MODELS.items():
    path = f'{ROOT}/results/mohler_{tag}.csv'
    done = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame(columns=['uid'])
    if len(done) >= len(mo):
        print(tag, 'already complete'); continue
    model, tok = load_hf_4bit(name)
    cand, _ = candidate_ids(tok, SCORE_TARGETS)
    texts = to_chat(tok, prompts) if chat else prompts
    add_special = not chat                      # the chat template already contains <|begin_of_text|>

    # sanity check: prompt tokens are a prefix of prompt+target, and the target is one token
    p_ids = tok(texts[0], add_special_tokens=add_special)['input_ids']
    full = tok(texts[0] + '7', add_special_tokens=add_special)['input_ids']
    assert full[:len(p_ids)] == p_ids and full[len(p_ids):] == [cand[7]], 'tokenisation mismatch'

    # latency / memory benchmark on 64 answers (batch 1 and batch 8)
    torch.cuda.reset_peak_memory_stats()
    for bs in (1, 8):
        torch.cuda.synchronize(); t = time.perf_counter()
        candidate_logprobs(model, tok, texts[:64], cand, batch_size=bs, add_special_tokens=add_special)
        torch.cuda.synchronize()
        latency.append({'model': tag, 'batch_size': bs, 'ms_per_answer': 1000 * (time.perf_counter() - t) / 64,
                        'peak_vram_gb': torch.cuda.max_memory_allocated() / 1e9})

    done_ids = set(done.uid)
    todo = [i for i in range(len(mo)) if mo.uid[i] not in done_ids]
    for start in tqdm(range(0, len(todo), CHUNK), desc=tag):
        idx = todo[start:start + CHUNK]
        lp = candidate_logprobs(model, tok, [texts[i] for i in idx], cand, batch_size=8, add_special_tokens=add_special)
        exp, arg, mass = score_from_logprobs(lp)
        chunk = pd.DataFrame({'uid': mo.uid.values[idx], 'expected': exp, 'argmax': arg, 'valid_mass': mass})
        chunk = pd.concat([chunk, pd.DataFrame(lp, columns=[f'lp_{s}' for s in SCORE_TARGETS])], axis=1)
        chunk.to_csv(path, mode='a', header=not os.path.exists(path), index=False)
    del model; gc.collect(); torch.cuda.empty_cache()

lat_path = f'{ROOT}/results/nb1_llm_latency.csv'
if latency:   # append, so a resumed session keeps the first model's measurement
    pd.DataFrame(latency).to_csv(lat_path, mode='a', header=not os.path.exists(lat_path), index=False)
pd.read_csv(lat_path).round(2) if os.path.exists(lat_path) else None

config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

llama3_instruct_zs:   0%|          | 0/9 [00:00<?, ?it/s]

,model,batch_size,ms_per_answer,peak_vram_gb
0,llama3_base_zs,1,1130.74,5.88
1,llama3_base_zs,8,1044.30,6.37
2,llama3_instruct_zs,1,1376.43,5.89
3,llama3_instruct_zs,8,1092.42,6.40


## 2. Add to the prediction table: raw scores, and scores rescaled per fold (linear fit on that fold's train+val)

In [5]:
preds = pd.read_csv(f'{ROOT}/results/mohler_preds.csv')
M = mo[['uid'] + [f'split_f{k}' for k in range(N_FOLDS)]]
diag = []
for tag in MODELS:
    z = pd.read_csv(f'{ROOT}/results/mohler_{tag}.csv').drop_duplicates('uid')
    d = preds[['uid', 'y', 'fold']].merge(z[['uid', 'expected', 'valid_mass']], on='uid').merge(M, on='uid')
    assert len(d) == len(preds)
    cal = pd.Series(np.nan, index=d.index)
    for k in range(N_FOLDS):
        fit, te = d[d[f'split_f{k}'] != 'test'], d[d[f'split_f{k}'] == 'test']
        lr = LinearRegression().fit(fit[['expected']], fit.y)
        cal[te.index] = np.clip(lr.predict(te[['expected']]), 0, 10)
    preds = preds.drop(columns=[tag, f'{tag}_cal'], errors='ignore').merge(
        d[['uid']].assign(**{tag: d.expected.values, f'{tag}_cal': cal.values}), on='uid')
    diag.append({'model': tag, 'mean_valid_mass': d.valid_mass.mean(), 'min_valid_mass': d.valid_mass.min(),
                 'pred_mean': d.expected.mean(), 'pred_std': d.expected.std()})
preds.to_csv(f'{ROOT}/results/mohler_preds.csv', index=False)
pd.DataFrame(diag).round(3)

,model,mean_valid_mass,min_valid_mass,pred_mean,pred_std
0,llama3_base_zs,0.799,0.546,6.616,0.857
1,llama3_instruct_zs,0.001,0.000,5.809,2.073


`valid_mass` = probability the model puts on the tokens 0–10. Near 1 means it understood the answer format; a low value for the base model is itself a finding worth reporting.

In [6]:
rows = []
for s in [c for c in preds.columns if c.startswith('llama3_')]:
    rows.append({'system': s, **summarise(preds.y, preds[s], reg_metrics, groups=preds.group, folds=preds.fold)})
nb1 = pd.DataFrame(rows)
nb1.to_csv(f'{ROOT}/results/mohler_nb1_summary.csv', index=False)
ref = pd.read_csv(f'{ROOT}/results/mohler_nb0_summary.csv')
cols = ['system', 'pearson', 'pearson_ci', 'spearman', 'mae', 'rmse', 'qwk', 'pearson_fold_std']
pd.concat([ref[ref.system.isin(['mean', 'minilm_cos', 'composite_fixed_cal', 'human_grader1_vs_grader2'])], nb1])[cols].round(3)

,system,pearson,pearson_ci,spearman,mae,rmse,qwk,pearson_fold_std
0,mean,NaN,NaN,NaN,1.803,2.215,NaN,NaN
2,minilm_cos,0.486,"[0.425, 0.545]",0.478,1.474,1.935,0.375,0.093
4,composite_fixed_cal,0.548,"[0.491, 0.601]",0.565,1.388,1.856,0.433,0.086
10,human_grader1_vs_grader2,0.597,NaN,0.601,1.496,2.621,0.490,NaN
0,llama3_base_zs,0.585,"[0.537, 0.633]",0.611,2.260,2.522,0.243,0.046
1,llama3_base_zs_cal,0.611,"[0.556, 0.663]",0.608,1.349,1.767,0.498,0.058
2,llama3_instruct_zs,0.643,"[0.593, 0.689]",0.605,2.703,3.106,0.372,0.078
3,llama3_instruct_zs_cal,0.640,"[0.587, 0.689]",0.602,1.265,1.701,0.568,0.081


## 3. A few examples (for a sanity check, not for the paper)

In [7]:
ex = mo[['uid', 'question', 'reference_answer', 'student_answer', 'y']].merge(preds[['uid'] + list(MODELS)], on='uid')
ex.sample(5, random_state=1).round(2)

,uid,question,reference_answer,student_answer,y,llama3_base_zs,llama3_instruct_zs
1298,m1298,What are the two main functions defined by a s...,push and pop,pop and push,10.00,7.05,7.60
107,m107,Where do C++ programs begin to execute?,At the main function.,main function,10.00,7.87,6.77
1087,m1087,What is a linked list?,A collection of elements that can be allocated...,"A series of ""Nodes"" structures with a value an...",9.00,7.11,6.56
1156,m1156,How are linked lists passed as arguments to a ...,By reference.,Pass-by-reference,10.00,8.05,7.80
1907,m1907,What is the difference between an array that i...,The static arrays are intialized only once whe...,The static array will be available to the whol...,7.25,6.29,5.42


In [8]:
versions['peft'] = __import__('peft').__version__
json.dump(versions, open(f'{ROOT}/results/env_nb1.json', 'w'), indent=2)
versions

{'python': '3.13.15',
 'torch': '2.11.0+cu128',
 'transformers': '5.16.1',
 'bitsandbytes': '0.50.2',
 'gpu': 'Tesla T4',
 'peft': '0.21.0'}